# LIV 캐싱: VLA forward → attention 서브행렬 + vision feature 저장

`research/data_generation/generate_dataset.py`(로컬 Ubuntu, robosuite 렌더링)가 만든
`manifest.jsonl` + 이미지를 읽어서, 각 이미지를 OFT 체크포인트로 forward하고
action→vision attention 서브행렬 + vision token feature를 `.npz`로 캐싱한다.
이후 LIVModule 학습(Phase 1)은 7B forward 없이 이 캐시만 반복해서 읽으면 된다.

**실행 전**: 런타임 유형을 GPU로 설정 (런타임 > 런타임 유형 변경 > GPU. 가능하면 A100, 없으면 L4/T4).

**이 노트북을 돌리기 전에 로컬 Ubuntu에서 먼저 해야 할 일**:
```bash
cd research/data_generation
python generate_dataset.py --task_suite_name libero_spatial --num_tasks 3 --groups_per_task 5 \
    --output_dir dataset_out
zip -r dataset_out.zip dataset_out
```
그 다음 아래 "2. 데이터 업로드" 셀에서 `dataset_out.zip`을 업로드한다.

In [ ]:
!nvidia-smi

## 1. 설치 (research/phase0.ipynb와 동일한 검증된 순서)

In [ ]:
!pip install -q torch torchvision torchaudio

In [ ]:
!git clone https://github.com/airhood/openvla-ivm.git
%cd openvla-ivm
!pip install -q -e .

`torch==2.2.0`은 NumPy 2.0 이전 ABI로 빌드되어 있는데, Colab은 NumPy 2.x가 기본으로 깔려 있어
`Failed to initialize NumPy: _ARRAY_API not found` 경고/오류가 날 수 있다. numpy를 1.x로 고정한다.

**아래 셀 실행 후에도 numpy 관련 경고/에러가 계속 뜨면**: 런타임 > 세션 다시 시작 → 이 노트북을
처음(nvidia-smi)부터 다시 순서대로 실행.

In [ ]:
!pip install -q "numpy<2"

이 스크립트는 라이브 렌더링을 하지 않지만(이미지는 이미 로컬에서 다 렌더링됨),
`run_libero_eval.py`가 모듈 최상단에서 `libero`를 import하기 때문에 LIBERO 설치 자체는 필요하다.

In [ ]:
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
!pip install -q -e LIBERO
!pip install -q -r experiments/robot/libero/libero_requirements.txt

robosuite/gym 등 LIBERO 쪽 설치가 numpy를 다시 2.x로 끌어올릴 수 있다. 다시 한번 고정한다.

In [ ]:
!pip install -q "numpy<2"

## 2. 데이터 업로드

로컬 Ubuntu에서 만든 `dataset_out.zip`(manifest.jsonl + 이미지)을 업로드한다.

In [ ]:
from google.colab import files
uploaded = files.upload()  # dataset_out.zip 선택

In [ ]:
import zipfile
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(".")
!ls dataset_out | head
!wc -l dataset_out/manifest.jsonl

## 3. 캐싱 실행

체크포인트는 Phase 0/LIV 배선 검증과 동일한 사전학습 체크포인트를 그대로 쓴다.
`manifest.jsonl`의 모든 행이 같은 task_suite여야 한다(unnorm_key 해석 때문 —
`generate_dataset.py`를 task_suite별로 따로 돌렸다면 캐싱도 task_suite별로 따로 실행).

`--n_extract_layers 8`은 ablation 후보(1/4/8) 중 최댓값 — 캐시 재생성 없이 여러 L 값을 시도할 수 있다.

In [ ]:
!python research/data_generation/build_liv_cache.py \
  --pretrained_checkpoint moojink/openvla-7b-oft-finetuned-libero-spatial \
  --manifest dataset_out/manifest.jsonl \
  --output_dir liv_cache_out \
  --n_extract_layers 8

## 4. 결과 확인

In [ ]:
import numpy as np
import json

rows = [json.loads(l) for l in open("liv_cache_out/cache_manifest.jsonl")]
print(f"{len(rows)}개 캐싱됨")
print(rows[0])

sample = np.load("liv_cache_out/" + rows[0]["cache_path"])
print("submatrix shape (L, H, N):", sample["submatrix"].shape)
print("vision_features shape (N, D):", sample["vision_features"].shape)
print("submatrix finite:", np.isfinite(sample["submatrix"]).all())
print("vision_features finite:", np.isfinite(sample["vision_features"]).all())

**판단 기준**: 캐싱된 개수가 `dataset_out/manifest.jsonl` 행 수와 (거의) 같고,
`submatrix`/`vision_features`가 NaN/Inf 없이 예상 shape(`(8, 32, 256)` / `(256, 4096)`)로
나오면 통과. `cache_manifest.jsonl`의 `object_pos`/`object_quat`가 원본 manifest와 일치하는지도
육안으로 확인.

## 5. 결과 다운로드

In [ ]:
!zip -rq liv_cache_out.zip liv_cache_out
from google.colab import files
files.download("liv_cache_out.zip")